<a href="https://colab.research.google.com/github/AstronomekDomansky/-.-1/blob/main/%D0%9F%D1%80%D0%B0%D0%BA%D1%82%D0%B8%D1%87%D0%B5%D1%81%D0%BA%D0%B0%D1%8F_16_%D0%9A%D0%BB%D0%B0%D1%81%D1%81%D0%B8%D1%87%D0%B5%D1%81%D0%BA%D0%BE%D0%B5_%D0%BC%D0%B0%D1%88%D0%B8%D0%BD%D0%BD%D0%BE%D0%B5_%D0%BE%D0%B1%D1%83%D1%87%D0%B5%D0%BD%D0%B8%D0%B5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Практическая работа 16. Классическое машинное обучение

Практическое задание №2. **Реализация модели классического машинного обучения.**

Работа выполняется на сгенерированной таблице службы поддержки (`df_gen`) или на своём CSV (`df`). Дальше одни и те же ячейки: обработка, обучение, метрики. Вторая модель — другой тип из трёх. Две модели одного типа второй работой не считаются.

В зависимости от выбранной задачи нужно реализовать одно из следующего. Для 12 баллов — два разных типа и сравнение.

1. **Регрессия** — прогноз числовой величины. Качество: MAE, RMSE, R². Разбор: значимость факторов и точность прогноза в разных диапазонах значений цели.
2. **Классификация** — отнесение объекта к категории. Качество: Accuracy, Precision, Recall, F1, AUC-ROC. Разбор: какие признаки важны для решения и какого рода ошибки делает модель (ложные срабатывания и пропуски).
3. **Кластеризация** — группы похожих объектов. Качество: Silhouette и рисунок групп. Разбор: размер кластеров, чем группы отличаются и какой у них практический смысл.

**Баллы**

| Что сдаётся | Баллы |
|---|---|
| Обработка данных, одна модель, её метрики и устная защита | 8 |
| Вторая модель другого типа, её метрики, сравнение с первой и устная защита | ещё 4, вместе 12 |
| Ответы на теоретические вопросы из банка в конце ноутбука | 4 |

Итого 16. На защите проговорите цель, признаки, подготовку и то, что показали метрики.


## 0. Фамилия, имя, номер в списке

От этих трёх полей зависит зерно генератора. Свой файл от зерна не зависит, но поля всё равно заполните: они нужны для ведомости.


In [ ]:
# ==========================================================
# ЗАПОЛНИТЕ ЭТИ ТРИ ПОЛЯ
# ==========================================================
last_name   = "Иванов"
first_name  = "Иван"
list_number = 1
# ==========================================================

import hashlib

def make_seed(last: str, first: str, number: int) -> int:
    key = f"{last.strip().lower()}|{first.strip().lower()}|{int(number)}"
    return int(hashlib.sha256(key.encode("utf-8")).hexdigest(), 16) % 100_000

SEED = make_seed(last_name, first_name, list_number)
print("Студент:", last_name, first_name, f"(№{list_number})")
print("SEED:", SEED)


Студент: Иванов Иван (№1)
SEED: 83962


In [ ]:
%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    r2_score, mean_absolute_error, mean_squared_error,
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    silhouette_score,
)
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

pd.set_option("display.max_columns", 40)
plt.style.use("seaborn-v0_8")


## 1. Откуда берётся таблица

Выполните **один** путь.

Путь А кладёт таблицу в `df_gen`. После генерации ячейка печатает, какие столбцы можно взять целью регрессии, какие — целью классификации, и какие числовые столбцы годятся для кластеризации. Скрытая метка сегмента в таблицу не входит.

Путь Б — свой CSV в переменной `df`. Дальше в ячейках обработки и моделей работайте с `df` так же, как с `df_gen`.

| Столбец `df_gen` | Смысл |
|---|---|
| `employees`, `servers` | размер компании и число серверов |
| `tickets_month` | обращений в поддержку за месяц |
| `monthly_rub` | платёж за поддержку |
| `remote_share` | доля удалённых сотрудников |
| `cpu_load` | средняя загрузка CPU, % |
| `reply_hours` | среднее время первого ответа, часы |
| `downtime_hours` | простой за месяц, часы |
| `fail_rate` | доля неуспешных изменений |
| `nps` | оценка клиентов |
| `contract` | срок договора: месяц, квартал, год |
| `channel` | канал обращения: телефон, чат, почта |
| `hours_support` | часы поддержки за месяц |
| `sla_breached` | 1 — SLA сорван, 0 — удержан |

В нескольких столбцах, включая `contract` и `channel`, пропуски занимают заметную долю строк. Срок договора и канал немного сдвигают и часы поддержки, и срыв SLA: без кодирования этих столбцов метрики хуже. У регрессии изредка бывают крупные промахи, поэтому MAE и RMSE расходятся. Доля класса 1 заметно ниже половины, поэтому Accuracy, Precision, Recall, F1 и AUC-ROC описывают модель по-разному.

Генератор — гауссовская копула с четырьмя скрытыми сегментами. Так же устроен синтезатор GaussianCopula в библиотеке SDV. Здесь это сделано на `numpy` и `scipy`.


### Путь А. Сгенерированная таблица `df_gen`


In [ ]:

import numpy as np
import pandas as pd
from scipy.stats import norm

def generate_support(seed: int, n: int = 1400) -> pd.DataFrame:
    """Гауссовская копула со скрытыми сегментами компаний.

    Тот же приём, что у GaussianCopula в библиотеке SDV: скрытые нормальные
    величины проходят через корреляцию и затем через свои распределения.
    Четыре сегмента задают кластеры. Цели собираются из части признаков с шумом.
    """
    rng = np.random.default_rng(seed)
    d = 6
    means = np.array([
        [-1.4, -1.2, -1.8, -1.3, -1.1, -1.8],
        [-1.4, -1.2, -1.8, -1.3,  1.1,  1.8],
        [ 1.4,  1.2,  1.8,  1.3, -1.1, -1.8],
        [ 1.4,  1.2,  1.8,  1.3,  1.1,  1.8],
    ])
    corr = np.eye(d)
    corr[0, 1] = corr[1, 0] = 0.45
    corr[2, 3] = corr[3, 2] = 0.35
    chol = np.linalg.cholesky(corr)
    labels = rng.integers(0, 4, size=n)
    z = means[labels] + 0.55 * (rng.normal(size=(n, d)) @ chol.T)
    u = norm.cdf(z)

    def q(col, lo, hi, how="uniform"):
        if how == "log":
            return np.exp(np.log(lo) + u[:, col] * (np.log(hi) - np.log(lo)))
        return lo + u[:, col] * (hi - lo)

    employees = q(0, 8, 800, "log").round().astype(int)
    servers = q(1, 2, 220, "log").round().astype(int)
    tickets = q(2, 20, 2400, "log").round().astype(int)
    monthly = q(3, 80_000, 4_500_000, "log").round(-2)
    remote = np.clip(q(4, 0.05, 0.95), 0.02, 0.98)
    cpu = np.clip(q(5, 15, 92), 5, 98)
    reply = np.clip(4 + 0.35 * z[:, 2] + 0.8 * z[:, 4] + rng.normal(0, 0.7, n), 0.4, 48)
    downtime = np.clip(np.exp(0.4 + 0.45 * z[:, 1] + 0.25 * z[:, 5] + rng.normal(0, 0.25, n)), 0.1, 40)
    fail = np.clip(0.08 + 0.04 * z[:, 5] + 0.03 * z[:, 2] + rng.normal(0, 0.02, n), 0.01, 0.45)
    nps = np.clip(8.2 - 0.9 * z[:, 4] - 0.4 * z[:, 2] + rng.normal(0, 0.6, n), 1, 10)
    contract_name = np.array(["месяц", "квартал", "год"])
    channel_name = np.array(["телефон", "чат", "почта"])
    contract_i = rng.integers(0, 3, n)
    channel_i = rng.integers(0, 3, n)
    noise = rng.normal(0, 16, n)
    spike = rng.random(n) < 0.05
    noise[spike] += rng.choice([-1.0, 1.0], size=int(spike.sum())) * rng.uniform(45, 90, size=int(spike.sum()))
    hours = (
        12 + 0.045 * tickets + 1.6 * reply + 2.2 * downtime + 0.03 * employees
        + np.array([22.0, 6.0, -12.0])[contract_i]
        + np.array([14.0, 0.0, -9.0])[channel_i]
        + noise
    )
    hours = np.clip(hours, 5, None)
    logit = (
        -2.7
        + 0.15 * reply + 0.12 * downtime + 3.2 * fail + 0.02 * (cpu - 40)
        + np.array([1.15, 0.1, -0.95])[contract_i]
        + np.array([0.75, 0.0, -0.55])[channel_i]
    )
    sla = rng.random(n) < 1 / (1 + np.exp(-logit))
    df = pd.DataFrame({
        "employees": employees,
        "servers": servers,
        "tickets_month": tickets,
        "monthly_rub": monthly,
        "remote_share": np.round(remote, 3),
        "cpu_load": np.round(cpu, 1),
        "reply_hours": np.round(reply, 2),
        "downtime_hours": np.round(downtime, 2),
        "fail_rate": np.round(fail, 3),
        "nps": np.round(nps, 2),
        "contract": contract_name[contract_i],
        "channel": channel_name[channel_i],
        "hours_support": np.round(hours, 1),
        "sla_breached": sla.astype(int),
    })

    def poke(col, rate):
        df.loc[rng.choice(n, size=int(rate * n), replace=False), col] = np.nan

    poke("nps", 0.14)
    poke("reply_hours", 0.10)
    poke("downtime_hours", 0.09)
    poke("tickets_month", 0.08)
    poke("employees", 0.07)
    poke("contract", 0.12)
    poke("channel", 0.11)
    return df

df_gen = generate_support(SEED)

numeric = df_gen.select_dtypes(include="number")
reg_cols = [c for c in numeric.columns if numeric[c].nunique(dropna=True) > 8]
cls_cols = [c for c in numeric.columns if numeric[c].nunique(dropna=True) <= 8]
cls_cols += df_gen.select_dtypes(exclude="number").columns.tolist()



In [ ]:
df_gen.head()

,employees,servers,tickets_month,monthly_rub,remote_share,cpu_load,reply_hours,downtime_hours,fail_rate,nps,contract,channel,hours_support,sla_breached
0,508.0,149,1339.0,3535200.0,0.440,16.2,4.72,2.07,0.036,8.19,квартал,почта,99.9,0
1,9.0,2,NaN,108900.0,0.283,15.3,2.25,0.31,0.010,10.00,квартал,чат,42.5,0
2,13.0,7,21.0,87600.0,0.138,30.0,2.83,1.25,0.010,10.00,месяц,почта,44.7,0
3,12.0,13,20.0,97000.0,0.664,86.6,4.03,1.78,0.010,7.51,квартал,телефон,54.2,1
4,23.0,3,27.0,145900.0,0.941,88.6,5.56,1.71,0.112,6.83,квартал,телефон,32.0,1


In [ ]:
print("df_gen:", df_gen.shape)
print("Пропуски:", int(df_gen.isna().sum().sum()))
print()
print("Можно взять целью регрессии:")
for col in reg_cols:
    print(f"  {col}  — различных значений: {int(numeric[col].nunique(dropna=True))}")
print()
print("Можно взять целью классификации:")
for col in cls_cols:
    print(f"  {col}  — различных значений: {int(df_gen[col].nunique(dropna=True))}")
cluster_cols = [c for c in numeric.columns if c not in cls_cols]
print()
print("Кластеризация обходится без цели. Числовые столбцы для групп:")
print(" ", ", ".join(cluster_cols))


df_gen: (1400, 14)
Пропуски: 994

Можно взять целью регрессии:
  employees  — различных значений: 449
  servers  — различных значений: 206
  tickets_month  — различных значений: 541
  monthly_rub  — различных значений: 1237
  remote_share  — различных значений: 609
  cpu_load  — различных значений: 297
  reply_hours  — различных значений: 492
  downtime_hours  — различных значений: 458
  fail_rate  — различных значений: 254
  nps  — различных значений: 405
  hours_support  — различных значений: 906

Можно взять целью классификации:
  sla_breached  — различных значений: 2
  contract  — различных значений: 3
  channel  — различных значений: 3

Кластеризация обходится без цели. Числовые столбцы для групп:
  employees, servers, tickets_month, monthly_rub, remote_share, cpu_load, reply_hours, downtime_hours, fail_rate, nps, hours_support


In [ ]:
# ▼▼▼ ВАШ КОД — путь А
# работа с датасетом в переменной df_gen
# выше параметры, с которыми можно работать

### Путь Б. Свой файл `df`

Откуда брать датасет:

- [Kaggle Datasets](https://www.kaggle.com/datasets)
- [Google Dataset Search](https://datasetsearch.research.google.com/)
- [Hugging Face Datasets](https://huggingface.co/datasets)

На странице датасета есть готовый кусок кода. Его копируют в ячейку и результат кладут в `df`.

На Kaggle это кнопка с кодом загрузки, часто через `kagglehub.dataset_download`. Дальше из скачанной папки читают CSV: `pd.read_csv(...)`. Если на странице виден прямой адрес файла `.csv`, его передают в `pd.read_csv` так же, как URL в практике 1.

На Hugging Face блок называется Use this dataset. Оттуда берут, например, `pd.read_csv("hf://datasets/...")` или `load_dataset(...)`, и таблицу снова записывают в `df`.

Строк — минимум сотни, столбцов — несколько. Для регрессии нужен числовой столбец с многими разными значениями. Для классификации — столбец с небольшим числом категорий, и каждая категория в таблице встречается. Для кластеризации — несколько числовых столбцов про объекты. Номер строки, id и уникальное имя объектом модели не являются: их не ставят ни в цель, ни в признаки. В описании датасета можно проверить что он содержит.


In [ ]:
# ▼▼▼ ВАШ КОД — путь Б, ячейка 1
# Прямой CSV или файл на диске, как в практике 1:
# df = pd.read_csv("https://....csv")
# df = pd.read_csv("ваш_файл.csv")
#
# Либо вставьте сюда кусок кода со страницы датасета и положите таблицу в df.
# print(df.shape)
# df.head()


## 2. Обработка данных

Эта ячейка входит в 8 баллов. Для сгенерированной таблицы работайте с `df_gen`. Для своего файла — с `df`.

1. Назначьте `table` (`df_gen` или `df`), тип задачи и имя цели. Для кластеризации цель не нужна.
2. Составьте `FEATURES`. Целевой столбец в этот список не входит
3. Предобработайте датасет в зависимости от задачи


In [ ]:
# ▼▼▼ ВАШ КОД — обработка



## 3. Первая модель

### 3.1. Обучение

По карточке выбранной задачи.

- Регрессия и классификация: `test_size=0.2`, `random_state=SEED`. Для классификации разбиение стратифицируйте. `Scaler` обучайте только на тренировочных строках.
- Кластеризация: масштабирование по `FEATURES`. Целевой столбец в матрицу не входит.
- Использовать способы улучшения метрик


In [ ]:
# ▼▼▼ ВАШ КОД — обучение первой модели


### 3.2. Результат и метрики

Сдайте то, что относится к вашей задаче.

**Регрессия.** На тесте MAE, RMSE и R². Scatter «факт — прогноз». MAE по нескольким диапазонам фактической цели. Коэффициенты и фраза, какие признаки тянут прогноз.

**Классификация.** Accuracy, Precision, Recall, F1, AUC-ROC. Матрица ошибок. Фраза, каких ошибок больше — ложных срабатываний или пропусков — и чем они отличаются для этой цели. Коэффициенты или другая картина вклада признаков.

**Кластеризация.** Силуэт для выбранного k, цвет — метка кластера. Размеры кластеров и средние признаки по группам. Чем группы отличаются и как их назвать.


In [ ]:
# ▼▼▼ ВАШ КОД — метрики и разбор первой модели


## 4. Вторая модель

Обучение модели. Вместе с сравнением и устной защитой это **12 баллов**.

### 4.1. Обучение

Та же подготовка таблицы. Своё разбиение и свой масштаб, если задача с учителем.


In [ ]:
# ▼▼▼ ВАШ КОД — обучение второй модели


### 4.2. Метрики второй модели

Тот же набор, что в §3.2, для типа второй модели.


In [ ]:
# ▼▼▼ ВАШ КОД — метрики второй модели


### 4.3. Сравнение с первой моделью

Эта ячейка обязательна, если сдаёте вторую модель.

Поставьте метрики первой и второй модели рядом. Если обе с учителем, считайте их на одном и том же тестовом разбиении. Напишите, где выводы совпали, что вторая модель добавляет к первой и какую из двух уместнее использовать для выбранного вопроса по таблице.


In [ ]:
# ▼▼▼ ВАШ КОД — сравнение двух моделей



## Банк теоретических вопросов

На защите преподаватель выбирает вопросы из этого списка. За ответы **4 балла**. Вопросы покрывают путь от таблицы до вывода, по темам лекций.

##Вопросы для сдачи работы:

#Блок 1. (Лекция 4) Линейная и полиномиальная регрессия, регуляризация

• Чем задача регрессии отличается от классификации? Что именно предсказывает регрессионная модель?

• Запишите уравнение линейной регрессии для нескольких признаков. Какой смысл имеют веса модели и свободный коэффициент?

• Какую задачу решает метод наименьших квадратов (МНК)? Почему в нём минимизируют именно квадраты ошибок?

• Запишите функцию потерь MSE. Чем она отличается от RSS и RMSE? Как они связаны между собой?

• Чем MAE отличается от MSE с точки зрения влияния выбросов? В какой ситуации MAE предпочтительнее?

• Что такое градиент функции потерь? Как градиентный спуск изменяет веса линейной регрессии?

• Почему большой learning rate может мешать обучению? Что произойдёт при слишком маленьком значении?

• Что означает коэффициент детерминации R^2? Может ли он быть отрицательным и что это означает?

• Если есть два признака x1, x2, какие дополнительные признаки могут появиться при использовании PolynomialFeatures(degree=2)?

• Почему увеличение степени полинома сначала может улучшать качество, а затем приводить к переобучению?

• Что такое переобучение и недообучение? Как они проявляются в значениях ошибки на train и validation?

• Для чего нужна регуляризация? Как изменяется функция потерь при добавлении L1- и L2-штрафов?

• В чём принципиальная разница между Ridge и Lasso? Почему Lasso может занулять некоторые коэффициенты?

• Что происходит с коэффициентами Ridge при увеличении параметра alpha? Что произойдёт при очень сильной регуляризации?

• Что такое Elastic Net?

• Почему при наличии сильно коррелирующих признаков коэффициенты линейной регрессии могут быть нестабильными? Как здесь помогает регуляризация?

#Блок 2. (Лекция 5) Классификация, логистическая регрессия, метрики и kNN

• Чем линейный классификатор отличается от линейной регрессии? Как выглядит разделяющая граница линейного классификатора в двумерном пространстве?

• Почему логистическая регрессия называется регрессией, хотя применяется для классификации?

• Запишите формулу сигмоиды. Почему её удобно использовать для преобразования выхода линейной модели в вероятность?

• Как в логистической регрессии определяется класс объекта по предсказанной вероятности? Обязательно ли порог должен равняться 0.5?

• Что такое функция потерь Log Loss (Binary Cross-Entropy)? Почему нельзя просто минимизировать количество неправильных классификаций?

• Что происходит со значением Log Loss, когда модель уверенно предсказывает неправильный класс?

• Для чего в логистической регрессии используется регуляризация? Как параметр C в sklearn связан с её силой?

• Что обозначают TP, TN, FP, FN в матрице ошибок? Приведите пример ложноположительной и ложноотрицательной ошибки.

• Как вычисляются Accuracy, Precision и Recall? Какая из этих метрик позволяет оценить количество пропущенных положительных объектов?

• В какой задаче важнее Precision, а в какой Recall?

• Как вычисляется F1-score? Почему используется гармоническое среднее, а не арифметическое?

• Что произойдёт с Precision и Recall, если снизить порог принятия решения с 0.5 до

• Чем ROC-AUC отличается от Accuracy? Почему ROC-AUC обычно рассчитывают по вероятностям или оценкам уверенности, а не по готовым меткам классов?

• В чём разница между micro-, macro- и weighted-усреднением F1-score при многоклассовой классификации? Какой вариант лучше обнаруживает проблемы с редкими классами?

• Как работает алгоритм kNN? Что фактически происходит во время вызова .fit() и во время .predict()?

• Как значение n_neighbors влияет на сложность границы классов и чувствительность к шуму? Что произойдёт при k=1?

#Блок 3. (Лекция 6) Деревья решений, случайный лес, бэггинг и бустинг

• Как строится дерево решений? Что представляют собой корневой узел, внутренние узлы и листья?

• По какому принципу дерево выбирает признак и порог для очередного разделения объектов?

• Что такое энтропия Шеннона? Запишите формулу. Когда энтропия набора объектов максимальна, а когда равна нулю?

• Как вычисляется индекс Джини (Gini impurity)? Чем он отличается от энтропии при выборе разбиения?

• Что такое Information Gain? Почему при расчёте качества разбиения нужно учитывать количество объектов в дочерних узлах?

• Как деревья решений решают задачу регрессии? Чем отличается построение регрессионного дерева от классификационного?

• Почему глубокое дерево легко переобучается? Как помогают параметры max_depth, min_samples_split и min_samples_leaf?

• Чем отличаются min_samples_split и min_samples_leaf? Почему это не взаимозаменяемые гиперпараметры?

• Почему деревья решений обычно не требуют стандартизации признаков? Какие преимущества и недостатки имеют деревья по сравнению с линейными моделями?

• Что такое ансамбль моделей? Почему несколько моделей могут давать более качественные предсказания, чем одна?

• Как работает бэггинг? Что такое bootstrap sampling и почему один объект может несколько раз попадать в обучающую подвыборку?

• Почему бэггинг обычно хорошо снижает дисперсию предсказаний? Почему важно, чтобы ошибки базовых моделей не были полностью одинаковыми?

• Чем Random Forest отличается от обычного бэггинга деревьев? Для чего при обучении каждого разбиения случайно выбирают подмножество признаков?

• Что означают параметры n_estimators, max_depth, max_features и bootstrap в RandomForestClassifier?

• Почему увеличение n_estimators в случайном лесе обычно не приводит к переобучению так же, как увеличение max_depth отдельного дерева?

• Что такое Out-of-Bag (OOB) оценка? Как её можно получить при обучении Random Forest?

• Каким образом случайный лес вычисляет важность признаков? Почему встроенные значения feature_importances_ не всегда стоит трактовать как причинное влияние признака?

• В чём принципиальная разница между бэггингом и бустингом? Почему модели в градиентном бустинге обучаются последовательно?

• Как работает AdaBoost? Каким образом меняются веса объектов после обучения очередного слабого классификатора?

• Чем XGBoost отличается от классического градиентного бустинга? Какую роль играют регуляризация и использование информации о второй производной функции потерь?

• Для чего в XGBoost нужны параметры reg_alpha, reg_lambda, subsample и colsample_bytree? Как они помогают бороться с переобучением?

• В чём заключается особенность построения деревьев в LightGBM? Чем стратегия leaf-wise отличается от level-wise и почему она может быстрее переобучаться?

• Что означает параметр num_leaves в LightGBM? Как он связан со сложностью деревьев и параметром max_depth?

• Почему CatBoost особенно удобен для категориальных признаков? Зачем нужны специальные способы обработки категорий и ordered boosting?

• Какие параметры вы стали бы настраивать в первую очередь в CatBoost: iterations, depth, learning_rate, l2_leaf_reg? За что отвечает каждый из них?

#Блок 4. (Лекция 7) Кластеризация и методы снижения размерности

• Чем обучение без учителя отличается от обучения с учителем? Почему в кластеризации заранее неизвестны правильные ответы?

• Что считается хорошим результатом кластеризации? Всегда ли объекты внутри одного кластера должны находиться близко друг к другу?

• Какие метрики расстояния используются для сравнения объектов? Запишите формулы евклидова, манхэттенского расстояний и расстояния Чебышёва.

• Как выбор метрики расстояния может изменить результат кластеризации? Почему два объекта могут быть близкими по одной метрике и далёкими по другой?

• Почему масштабирование признаков важно для алгоритмов кластеризации, основанных на расстояниях?

• Что называют проблемой проклятия размерности? Почему в многомерных пространствах становится сложнее определять ближайших соседей?

• Опишите основные шаги алгоритма K-Means. Как происходит назначение объектов кластерам и пересчёт центроидов?

• Почему K-Means чувствителен к начальному расположению центроидов? Какую проблему решает инициализация k-means++?

• Почему K-Means плохо работает с вытянутыми, кольцевыми или сильно различающимися по плотности кластерами

• Чем агломеративная кластеризация отличается от дивизимной? Как строится иерархия кластеров в каждом случае?

• Что такое дендрограмма? Как определить количество кластеров, проведя горизонтальный срез дендрограммы?

• В чём преимущества и недостатки иерархической кластеризации по сравнению с K-Means?

• Как работает DBSCAN? Что определяют параметры eps и min_samples?

• Чем отличаются core point, border point и noise point в DBSCAN? Может ли граничная точка не быть центральной, но всё равно принадлежать кластеру?

• Что произойдёт при слишком маленьком и слишком большом eps? Как это отразится на количестве кластеров и шумовых объектов?

• Для чего используются методы снижения размерности? Чем отличается снижение размерности для визуализации от преобразования признаков перед обучением?

• Как работает t-SNE на уровне общей идеи? Почему алгоритм преобразует расстояния между объектами в вероятности соседства?

• В чём идея линейного дискриминантного анализа (LDA)? Почему, в отличие от большинства рассматриваемых методов, LDA использует метки классов?

#Блок 5. (Лекция 8) Обработка естественного языка (NLP)

• Какие основные задачи решает NLP? Приведите примеры задач вида «текст → класс», «текст → последовательность меток» и «текст → текст».

• Какие этапы обычно включает классический пайплайн классификации текстов — от исходных документов до предсказания класса?

• Что такое токенизация? Чем токенизация по словам отличается от токенизации по символам или подсловам?

• Для чего выполняют нормализацию регистра, удаление пунктуации и специальных символов? Почему некоторые из этих операций могут ухудшить качество?

• В чём разница между стеммингом и лемматизацией? Какой подход учитывает морфологические свойства слов?

• Почему при лемматизации иногда необходимо учитывать контекст и часть речи?

• Что такое стоп-слова? Почему удаление слов «не», «нет», «без» может быть особенно опасным для анализа тональности?

• Что такое n-граммы? Чем биграммы отличаются от униграмм и какие недостатки подхода Bag-of-Words они позволяют частично компенсировать?

• Почему классические модели машинного обучения не могут напрямую работать со строками текста? Что понимают под векторизацией?

• Как работает Bag-of-Words? Как формируется словарь и что означает каждая координата полученного вектора?

• Какой главный недостаток Bag-of-Words связан с порядком слов? Почему предложения «собака укусила человека» и «человек укусил собаку» могут получить одинаковые представления?

• Что такое разреженная матрица признаков? Почему при векторизации большого корпуса текстов обычно не используют плотную матрицу NumPy?

• Что такое TF-IDF? Запишите общую формулу и объясните смысл компонентов TF и IDF.

• Что такое косинусная близость двух векторов? Запишите формулу. Почему для сравнения текстовых векторов часто используют косинусную близость, а не евклидово расстояние?

• Что такое эмбеддинг слова? Чем плотные векторные представления Word2Vec отличаются от разреженных векторов Bag-of-Words?

• Как работает Word2Vec на уровне обучающей задачи? Чем отличаются архитектуры CBOW и Skip-Gram

• В чём основная идея GloVe? Чем обучение на глобальной матрице совместной встречаемости отличается от использования локального контекста в Word2Vec?

• Что такое тематическое моделирование? Какую задачу решает Latent Dirichlet Allocation (LDA)?



Сдайте файл как `Фамилия_Имя_ПР16.ipynb`.

На защите: одна модель — 8 баллов, две модели разных типов вместе со сравнением — 12 баллов, теоретические вопросы из банка — 4 балла.
